# Annotation counts
For the statistics given in the manuscript.

In [96]:
from Bio import SeqIO
from collections import Counter, defaultdict
from tqdm.auto import tqdm
import numpy as np
import os
import pandas as pd

counts = defaultdict(lambda: Counter())

---
## Pfam annotations

### InterProScan

In [97]:
ipr_pfam_fn = '../data/functional_predictions/interproscan/Tomato_dataset.with_Pfam.txt.gz'
df = pd.read_csv(ipr_pfam_fn, sep='\t', names=['prot_id', 'pfam']).drop_duplicates('prot_id')
df['species'] = df['prot_id'].apply(lambda x: x[:5])

for sp, zdf in df.groupby('species'):
    counts['ipr_pfam'][sp] = len(zdf)

### EggNOG-mapper

In [98]:
eggnog_mapper_header = ['query',
 'seed_ortholog',
 'evalue',
 'score',
 'eggNOG_OGs',
 'max_annot_lvl',
 'COG_category',
 'Description',
 'Preferred_name',
 'GOs',
 'EC',
 'KEGG_ko',
 'KEGG_Pathway',
 'KEGG_Module',
 'KEGG_Reaction',
 'KEGG_rclass',
 'BRITE',
 'KEGG_TC',
 'CAZy',
 'BiGG_Reaction',
 'PFAMs']

def read_eggnog(sp):
    fn = '../data/functional_predictions/eggnog-mapper/' + sp + '/' + sp + '.emapper.annotations.gz'
    df = pd.read_csv(fn, sep='\t', comment='#', names=eggnog_mapper_header)
    return df

def pfam_id_formatter(x):
    if x.upper().startswith('PF'):
        return x.upper()
    else:
        return x.lower()

In [99]:
c_egg_pfam = []
for sp in tqdm(os.listdir('../data/functional_predictions/eggnog-mapper')):
    df = read_eggnog(sp)
    df['has_pfam'] = (df.PFAMs != '-')
    counts['egg_pfam'][sp] = df.has_pfam.sum()

  0%|          | 0/64 [00:00<?, ?it/s]

---
## GO

### InterProScan

In [100]:
ipr_go_fn = '../data/functional_predictions/interproscan/Tomato_dataset.with_GO.txt.gz'
df = pd.read_csv(ipr_go_fn, sep='\t', names=['prot_id', 'go_ids']).drop_duplicates('prot_id')
df['species'] = df['prot_id'].apply(lambda x: x[:5])

for sp, zdf in df.groupby('species'):
    counts['ipr_go'][sp] = len(zdf)

In [101]:
c_egg_go = []
for sp in tqdm(os.listdir('../data/functional_predictions/eggnog-mapper')):
    df = read_eggnog(sp)
    df['has_go'] = (df.GOs != '-')
    counts['egg_go'][sp] = df.has_go.sum()

  0%|          | 0/64 [00:00<?, ?it/s]

### FANTASIA

In [102]:
def read_fantasia(sp):
    fn = '../data/functional_predictions/fantasia/results/' + sp + '/results_' + sp + '.csv.gz'
    df = pd.read_csv(fn)
    return df

In [103]:
for sp in tqdm(os.listdir('../data/functional_predictions/fantasia/results')):
    df = read_fantasia(sp).drop_duplicates('query_accession')
    counts['fantasia_go'][sp] = len(df)

  0%|          | 0/64 [00:00<?, ?it/s]

---

## Results

In [111]:
for sp in tqdm(os.listdir('../data/proteomes/')):
    recs = list(SeqIO.parse('../data/proteomes/{}'.format(sp), 'fasta'))
    counts['total_proteins'][sp.split('.')[0]] = len(recs)

  0%|          | 0/64 [00:00<?, ?it/s]

In [112]:
def gen():
    species = sorted({sp for x in counts.values() for sp in x.keys()})
    for sp in species:
        r = {'species': sp}
        for k in counts.keys():
            r[k] = counts[k][sp]
        yield r

In [113]:
count_df = pd.DataFrame(gen())

In [114]:
count_df.to_csv('../data/functional_predictions/atleast_one_prediction.tsv', sep='\t', index=False)

In [121]:
for k in list(count_df.keys())[1:-1]:
    print(k, (count_df[k] / count_df['total_proteins']).mean() * 100)

ipr_pfam 69.47142780210396
egg_pfam 77.76466106093274
ipr_go 60.72127608892374
egg_go 42.251692011543575
fantasia_go 99.9999680620567


In [119]:
count_df

,species,ipr_pfam,egg_pfam,ipr_go,egg_go,fantasia_go,total_proteins
0,ARATH,28983,28921,25273,20701,35386,35386
2,CAPA2,27466,31869,25095,17595,35845,35845
4,CAPC2,27462,31013,24450,17122,34974,34974
6,COFEU,33125,34287,28453,18440,38150,38150
8,HEINZ,23866,26188,20837,14534,34075,34075
...,...,...,...,...,...,...,...
118,TS545,23078,27168,20157,14216,37305,37305
120,TS623,23875,27179,20964,14560,36207,36207
122,TS629,22472,26431,19849,13974,36569,36569
124,TS692,22722,26600,19812,14003,36578,36578
